In [ ]:
# %% [markdown]
# # Battle of Teams – интерактивная игра (ipywidgets)
# 
# Игра с поддержкой режимов PvP и PvE (против AI). ИИ использует обученную модель, если файл `environment/BattleOfTeams-v1.pt` существует.

# %%
import ipywidgets as widgets
from IPython.display import display, clear_output
import random
import numpy as np
import torch
from pathlib import Path
from dataclasses import dataclass
from typing import List, Optional
from PIL import Image
import io

# ==================== МОДЕЛЬ ДАННЫХ ====================
@dataclass
class Fighter:
    name: str
    strength: int
    alive: bool = True

@dataclass
class Team:
    name: str
    fighters: List[Fighter]

    def total_strength(self) -> int:
        return sum(f.strength for f in self.fighters if f.alive)

    def alive_count(self) -> int:
        return sum(1 for f in self.fighters if f.alive)

    def is_defeated(self) -> bool:
        return self.alive_count() == 0

class BattleModel:
    def __init__(self, team1: Team, team2: Team):
        self.team_agent = team1   # AI или первый игрок
        self.team_player = team2  # человек или второй игрок
        self.winner = None

    @staticmethod
    def create_random_team(name: str, total_strength: int = 100, num_fighters: int = 3) -> Team:
        strengths = []
        remaining = total_strength
        for i in range(num_fighters - 1):
            s = random.randint(1, max(1, remaining - (num_fighters - i - 1)))
            strengths.append(s)
            remaining -= s
        strengths.append(remaining)
        fighters = [Fighter(name=f"Боец {i+1}", strength=s) for i, s in enumerate(strengths)]
        return Team(name=name, fighters=fighters)

    def execute_round(self, attacker_team: Team, defender_team: Team, action: int):
        num_f = len(attacker_team.fighters)
        attacker_idx = action // num_f
        defender_idx = action % num_f

        alive_attackers = [i for i, f in enumerate(attacker_team.fighters) if f.alive]
        alive_defenders = [i for i, f in enumerate(defender_team.fighters) if f.alive]

        if not alive_attackers or not alive_defenders:
            return 0, 0

        if attacker_idx not in alive_attackers or defender_idx not in alive_defenders:
            attacker_idx = random.choice(alive_attackers)
            defender_idx = random.choice(alive_defenders)

        attacker = attacker_team.fighters[attacker_idx]
        defender = defender_team.fighters[defender_idx]

        attack_val = random.randint(0, attacker.strength)
        defense_val = random.randint(0, defender.strength)
        min_val = min(attack_val, defense_val)

        if attack_val > defense_val:
            attacker.strength += min_val
            defender.strength -= min_val
            if defender.strength <= 0:
                defender.alive = False
                defender.strength = 0
        elif attack_val < defense_val:
            defender.strength += min_val
            attacker.strength -= min_val
            if attacker.strength <= 0:
                attacker.alive = False
                attacker.strength = 0
        else:
            if random.random() < 0.5:
                attacker.strength += attack_val
                defender.strength -= attack_val
                if defender.strength <= 0:
                    defender.alive = False
                    defender.strength = 0
            else:
                defender.strength += defense_val
                attacker.strength -= defense_val
                if attacker.strength <= 0:
                    attacker.alive = False
                    attacker.strength = 0

        for f in attacker_team.fighters + defender_team.fighters:
            f.strength = max(0, int(f.strength))

        if attacker_team.is_defeated():
            self.winner = defender_team.name
        elif defender_team.is_defeated():
            self.winner = attacker_team.name

        return attack_val, defense_val

    def get_observation(self, for_agent_team: bool = True) -> List[float]:
        if for_agent_team:
            team_a = self.team_agent
            team_b = self.team_player
        else:
            team_a = self.team_player
            team_b = self.team_agent
        obs = []
        for team in (team_a, team_b):
            for fighter in team.fighters:
                obs.append(float(fighter.strength if fighter.alive else 0))
                obs.append(1.0 if fighter.alive else 0.0)
        return obs

# ==================== AI PLAYER ====================
class AIPlayer:
    def __init__(self, checkpoint_path: str = None):
        self.model = None
        if checkpoint_path and Path(checkpoint_path).exists():
            self.model = self._load_model(checkpoint_path)
            print(f"Модель загружена из {checkpoint_path}")
        else:
            print("Модель не найдена, AI будет выбирать случайные действия")

    def _load_model(self, path):
        class SLMCompatibleMLP(torch.nn.Module):
            def __init__(self, input_dim=12, hidden_dims=[128,128], output_dim=9):
                super().__init__()
                layers = []
                prev = input_dim
                for h in hidden_dims:
                    layers.append(torch.nn.Linear(prev, h))
                    layers.append(torch.nn.Tanh())
                    prev = h
                self.model = torch.nn.Sequential(*layers)
                self.tails = torch.nn.ModuleList([torch.nn.Linear(prev, output_dim)])
            def forward(self, x):
                x = self.model(x)
                x = self.tails[0](x)
                return x
        model = SLMCompatibleMLP()
        checkpoint = torch.load(path, map_location='cpu')
        state_dict = checkpoint.get('net', checkpoint)
        model.load_state_dict(state_dict)
        model.eval()
        return model

    def get_action(self, observation: list) -> int:
        if self.model is None:
            return random.randint(0, 8)
        obs = np.array(observation, dtype=np.float32)
        with torch.no_grad():
            tensor = torch.from_numpy(obs).unsqueeze(0)
            logits = self.model(tensor)
            action = torch.argmax(logits, dim=1).item()
        return action

# ==================== ИГРОВОЙ ИНТЕРФЕЙС (ipywidgets) ====================
class GameUI:
    def __init__(self):
        self.battle = None
        self.ai_player = None
        self.mode = None
        self.current_player = None
        self.selected_attacker = None
        self.buttons = {}  # хранение кнопок для обновления
        self.log_area = widgets.Output()
        self.status = widgets.HTML(value="<b>Выберите режим игры</b>")
        self.team1_box = widgets.VBox()
        self.team2_box = widgets.VBox()
        self.control_panel = widgets.VBox()

    def start(self):
        # Режим выбора
        mode_selector = widgets.RadioButtons(
            options=['PvE (Игрок vs AI)', 'PvP (Игрок1 vs Игрок2)'],
            description='Режим:',
            style={'description_width': 'initial'}
        )
        start_btn = widgets.Button(description='Начать игру', button_style='success')
        output = widgets.Output()

        def on_start(b):
            mode = 'pve' if mode_selector.value == 'PvE (Игрок vs AI)' else 'pvp'
            with output:
                clear_output()
                self.init_game(mode)
        start_btn.on_click(on_start)

        display(widgets.VBox([mode_selector, start_btn, output]))

    def init_game(self, mode):
        self.mode = mode
        self.selected_attacker = None

        team1 = BattleModel.create_random_team("Красные")
        team2 = BattleModel.create_random_team("Синие")

        if mode == 'pve':
            self.battle = BattleModel(team1, team2)
            self.ai_player = AIPlayer("game/environment/BattleOfTeams-v1.pt")  # путь к модели
            # случайный первый ход
            self.current_player = random.choice(['agent', 'player'])
        else:
            self.battle = BattleModel(team1, team2)
            self.current_player = random.choice(['player1', 'player2'])

        # Создаём интерфейс
        self.build_ui()

        if mode == 'pve' and self.current_player == 'agent':
            self.make_ai_move()

    def build_ui(self):
        # Очистить старые виджеты
        self.team1_box.children = []
        self.team2_box.children = []
        self.buttons = {}

        team_a = self.battle.team_agent
        team_b = self.battle.team_player

        # Создаём кнопки для каждой команды
        for team, side, box in [(team_a, 'agent', self.team1_box), (team_b, 'player', self.team2_box)]:
            for idx, fighter in enumerate(team.fighters):
                btn = widgets.Button(
                    description=f"{fighter.name}\nСила: {fighter.strength}",
                    layout=widgets.Layout(width='120px', height='100px'),
                    button_style='',
                    tooltip=f"Клик для выбора"
                )
                if not fighter.alive:
                    btn.disabled = True
                    btn.button_style = 'danger'
                btn.on_click(lambda b, s=side, i=idx: self.on_fighter_click(s, i))
                box.children += (btn,)
                self.buttons[(side, idx)] = btn

        # Панель управления и лог
        self.status.value = f"<b>Ход: {self.current_player.upper()}</b>"
        if self.mode == 'pve':
            info = "Игрок управляет синими, AI – красными"
        else:
            info = "Игрок1 – красные, Игрок2 – синие"

        self.control_panel.children = [
            widgets.HTML(f"<div style='margin:10px'>{info}</div>"),
            self.status,
            widgets.HBox([self.team1_box, self.team2_box]),
            self.log_area,
            widgets.Button(description='Сбросить игру', button_style='warning', on_click=self.reset_game)
        ]
        display(self.control_panel)

    def on_fighter_click(self, side, idx):
        if self.battle.winner:
            self.add_log("Игра уже завершена. Нажмите 'Сбросить игру'.")
            return

        # Проверка, чей ход
        allowed_attacker_side = None
        if self.mode == 'pve':
            if self.current_player == 'agent':
                self.add_log("Сейчас ход AI, подождите...")
                return
            allowed_attacker_side = 'player'
        else:  # pvp
            if self.current_player == 'player1':
                allowed_attacker_side = 'agent'
            else:
                allowed_attacker_side = 'player'

        fighter = self.get_fighter(side, idx)
        if not fighter or not fighter.alive:
            self.add_log("Этот боец мёртв, выберите живого.")
            return

        if self.selected_attacker is None:
            # Выбираем атакующего
            if side != allowed_attacker_side:
                self.add_log("Сейчас не ваш ход! Выберите атакующего из своей команды.")
                return
            self.selected_attacker = (side, idx)
            self.highlight_button(side, idx, True)
            self.add_log(f"Выбран атакующий: {fighter.name}")
        else:
            attacker_side, attacker_idx = self.selected_attacker
            if side == attacker_side:
                self.add_log("Нельзя атаковать своего бойца! Выберите защитника из команды противника.")
                return
            if not fighter.alive:
                self.add_log("Защитник мёртв, выберите другого.")
                return
            # Выполняем атаку
            action = attacker_idx * 3 + idx
            self.execute_round(attacker_side, idx, action)

    def execute_round(self, defender_side, defender_idx, action):
        # Определяем команды в зависимости от режима
        if self.mode == 'pve':
            attacker_team = self.battle.team_player
            defender_team = self.battle.team_agent
        else:  # pvp
            if self.current_player == 'player1':
                attacker_team = self.battle.team_agent
                defender_team = self.battle.team_player
            else:
                attacker_team = self.battle.team_player
                defender_team = self.battle.team_agent

        attacker_idx = action // 3
        defender_idx_actual = action % 3

        attacker = attacker_team.fighters[attacker_idx]
        defender = defender_team.fighters[defender_idx_actual]

        attack_val, defense_val = self.battle.execute_round(attacker_team, defender_team, action)

        log_msg = f"{attacker.name} ({attacker_team.name}) атакует {defender.name} ({defender_team.name}): "
        if attack_val > defense_val:
            log_msg += f"атака {attack_val} > защита {defense_val} → победа! {attacker.name} усиливается."
        elif attack_val < defense_val:
            log_msg += f"атака {attack_val} < защита {defense_val} → поражение! {attacker.name} ослабевает."
        else:
            log_msg += f"атака = защита → ничья, случайный исход."
        self.add_log(log_msg)

        # Обновляем UI
        self.update_ui()

        if self.battle.winner:
            self.add_log(f"🏆 ПОБЕДИТЕЛЬ: {self.battle.winner}!")
            self.status.value = f"<b>Победитель: {self.battle.winner}</b>"
            return

        # Переключение хода
        self.switch_turn()
        if self.mode == 'pve' and self.current_player == 'agent':
            self.make_ai_move()

    def switch_turn(self):
        if self.mode == 'pve':
            self.current_player = 'player' if self.current_player == 'agent' else 'agent'
        else:
            self.current_player = 'player1' if self.current_player == 'player2' else 'player2'
        self.status.value = f"<b>Ход: {self.current_player.upper()}</b>"
        self.selected_attacker = None
        self.clear_highlight()

    def make_ai_move(self):
        if self.battle.winner or self.current_player != 'agent':
            return
        obs = self.battle.get_observation(for_agent_team=True)
        action = self.ai_player.get_action(obs)
        # Проверка валидности действия (живые бойцы)
        valid_actions = []
        for a in range(3):
            if not self.battle.team_agent.fighters[a].alive:
                continue
            for d in range(3):
                if self.battle.team_player.fighters[d].alive:
                    valid_actions.append(a * 3 + d)
        if not valid_actions:
            return
        if action not in valid_actions:
            action = random.choice(valid_actions)
        attacker_idx = action // 3
        defender_idx = action % 3
        attacker = self.battle.team_agent.fighters[attacker_idx]
        defender = self.battle.team_player.fighters[defender_idx]
        attack_val, defense_val = self.battle.execute_round(self.battle.team_agent, self.battle.team_player, action)
        self.add_log(f"🤖 AI ({attacker.name}) атакует {defender.name}: {attack_val} vs {defense_val}")
        self.update_ui()
        if self.battle.winner:
            self.add_log(f"🏆 ПОБЕДИТЕЛЬ: {self.battle.winner}!")
            self.status.value = f"<b>Победитель: {self.battle.winner}</b>"
            return
        self.switch_turn()
        if self.current_player == 'agent':
            # Если после переключения ход снова AI (только если оба AI? но в PvE такого нет), но для PvE не должно.
            pass

    def update_ui(self):
        # Обновляем кнопки согласно новому состоянию бойцов
        for side in ('agent', 'player'):
            team = self.battle.team_agent if side == 'agent' else self.battle.team_player
            for idx, fighter in enumerate(team.fighters):
                btn = self.buttons.get((side, idx))
                if btn is None:
                    continue
                btn.description = f"{fighter.name}\nСила: {fighter.strength}"
                if not fighter.alive:
                    btn.disabled = True
                    btn.button_style = 'danger'
                else:
                    btn.disabled = False
                    btn.button_style = ''
        self.clear_highlight()

    def highlight_button(self, side, idx, highlight):
        btn = self.buttons.get((side, idx))
        if btn:
            if highlight:
                btn.button_style = 'success'
            else:
                btn.button_style = ''

    def clear_highlight(self):
        for (side, idx), btn in self.buttons.items():
            btn.button_style = ''

    def get_fighter(self, side, idx):
        if side == 'agent':
            return self.battle.team_agent.fighters[idx]
        else:
            return self.battle.team_player.fighters[idx]

    def add_log(self, msg):
        with self.log_area:
            print(msg)

    def reset_game(self, b):
        clear_output(wait=True)
        self.start()

# ==================== ЗАПУСК ====================
game = GameUI()
game.start()